<p style="font-size: 22px; font-weight: bold">[AL-DoE for Optimization of Liquid Electrolyte Composition]</p>

* Creator: Jaehyun Park
* Email: jhyuun@o.cnu.ac.kr

# Install package

In [ ]:
!pip3 install bayesian-optimization
!pip3 install pandas
!pip3 install numpy
!pip install openpyxl
import numpy as np
from numpy import *
import pandas as pd
import os
import sklearn
import itertools
import math
from sklearn.preprocessing import *
from sklearn.linear_model import *
from sklearn.gaussian_process.kernels import Matern
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import cross_val_score, KFold, cross_validate, GridSearchCV, RandomizedSearchCV, train_test_split
from matplotlib import pyplot as plt
from scipy.optimize import minimize_scalar
from scipy.stats import norm
import warnings

# Import experimental data from csv file

---
* **Data format** (※ Data file should be the "csv" format)

| PC content in solvent	| Type of additive | Additive content | SET	| Retention	| Final discharge capacity | Latter retention |
|:-:|:-:|:-:|:-:|:-:|:-:|:-:|
|0~4 (0.1 gap)|0~2 (1 gap)|0~3 (0.5 gap)|-|-|-|-|-|
---
* **Input representative**
| No. | Type of Additive |
|:-----:|:------------------:|
|  0  |         X        |
|  1  |        VC        |
|  2  |       FEC        |
---

In [ ]:
exp_data = pd.read_csv(r"") #initial experimetal dataset
display(exp_data)
exp_data = exp_data.sample(frac=1, random_state=2023).reset_index(drop=True)
display(exp_data)

# Extract needed dataset

In [ ]:
x_train = exp_data.iloc[:, :3] #input dataset
y_train = exp_data.iloc[:, 3:] #output dataset
display(x_train)
display(y_train)

# Split the output data for train each GP

In [ ]:
y_train1 = y_train.iloc[:, [0]]
y_train2 = y_train.iloc[:, [1]]
y_train3 = y_train.iloc[:, [2]]
y_train4 = y_train.iloc[:, [3]]

y_train1_std = (y_train1 - mean(y_train1, axis=0)) / std(y_train1, axis=0)
y_train2_std = (y_train2 - mean(y_train2, axis=0)) / std(y_train2, axis=0)
y_train3_std = (y_train3 - mean(y_train3, axis=0)) / std(y_train3, axis=0)
y_train4_std = (y_train4 - mean(y_train4, axis=0)) / std(y_train4, axis=0)

print(y_train1_std)
print(y_train2_std)
print(y_train3_std)
print(y_train4_std)

y_train1 = y_train1_std * std(y_train1, axis=0) + mean(y_train1, axis=0)
y_train2 = y_train2_std * std(y_train2, axis=0) + mean(y_train2, axis=0)
y_train3 = y_train3_std * std(y_train3, axis=0) + mean(y_train3, axis=0)
y_train4 = y_train4_std * std(y_train4, axis=0) + mean(y_train4, axis=0)

print(y_train1)
print(y_train2)
print(y_train3)
print(y_train4)

# Generate candidate matrix for experiment 

In [ ]:
search_space_final1 = np.array(list(itertools.product(np.arange(0, 4.1, 0.1), np.arange(1,3,1), np.arange(0.5, 3.1, 0.5))))
search_space_final2 = np.array(list(itertools.product(np.arange(0, 4.1, 0.1), np.arange(0,1,1), np.arange(0, 1, 1))))
search_space_final = np.concatenate((search_space_final1, search_space_final2), axis=0)

print(search_space_final)
datadata = pd.DataFrame(search_space_final)
print(datadata)
print(search_space_final.shape)

# Train and test GP model with hyperparameter tuning (1st iteration)

In [ ]:
def custom_r2_scorer(estimator, X, y):
    y_pred = estimator.predict(X)
    r2 = r2_score(y, y_pred)
    return r2

def custom_rmse_scorer(estimator, X, y):
    y_pred = estimator.predict(X)
    mse = mean_squared_error(y, y_pred)
    rmse = np.sqrt(mse)
    return rmse

global n_iter, cv
n_iter = 2000
cv = 4

## GP for SET

In [ ]:
kernel1 = Matern()

param_dist1 = {'alpha': np.logspace(-10,0,11),
               'kernel__length_scale': np.logspace(-5,5,11),
               'kernel__nu': np.linspace(0.5,10,20)
              }

model_gp1 = GaussianProcessRegressor(kernel= kernel1,
                                 alpha = 'alpha',
                                 optimizer=None,
                                 random_state=0)

search1 = RandomizedSearchCV(model_gp1, param_distributions=param_dist1,
                             n_iter = n_iter, cv = cv, scoring={'r2': custom_r2_scorer,'rmse': custom_rmse_scorer},  
                             n_jobs=-1, verbose=1, refit='r2')

search1.fit(x_train, y_train1_std)

print(search1.best_params_, '\n')

best_fold_index_r2 = np.argmax(search1.cv_results_['mean_test_r2'])

kf = KFold(n_splits=cv)
test_indices = []

for _, test_index in kf.split(x_train):
    test_indices.append(test_index)

average_r2 = search1.cv_results_['mean_test_r2'][best_fold_index_r2]
average_rmse = search1.cv_results_['mean_test_rmse'][best_fold_index_r2] 
    
for fold_index in range(cv):
    r2_fold = search1.cv_results_['split{}_test_r2'.format(fold_index)][best_fold_index_r2]
    rmse_fold = search1.cv_results_['split{}_test_rmse'.format(fold_index)][best_fold_index_r2]

    print(f"Fold {fold_index + 1} - R^2: {r2_fold}, RMSE: {rmse_fold}")

    test_indices_fold = test_indices[fold_index]

    print(f"Fold {fold_index + 1} - Test Indices: {test_indices_fold}", '\n')

print('\n', 'Average R2 score:', average_r2)
print('Average RMSE:', average_rmse)

best_gp1 = search1.best_estimator_

## GP for retention

In [ ]:
kernel2 = Matern()

param_dist2 = {'alpha': np.logspace(-10,0,11),
               'kernel__length_scale': np.logspace(-5,5,11),
               'kernel__nu': np.linspace(0.5,10,20)
              }


model_gp2 = GaussianProcessRegressor(kernel= kernel2,
                                 alpha = 'alpha',
                                 optimizer=None,
                                 random_state=0)

search2 = RandomizedSearchCV(model_gp2, param_distributions=param_dist2,
                             n_iter = n_iter, cv = cv, scoring={'r2': custom_r2_scorer,'rmse': custom_rmse_scorer},  
                             n_jobs=-1, verbose=1, refit='r2')

search2.fit(x_train, y_train2_std)

print(search2.best_params_, '\n')

best_fold_index_r2 = np.argmax(search2.cv_results_['mean_test_r2'])

kf = KFold(n_splits=cv)
test_indices = []

for _, test_index in kf.split(x_train):
    test_indices.append(test_index)

average_r2 = search2.cv_results_['mean_test_r2'][best_fold_index_r2]
average_rmse = search2.cv_results_['mean_test_rmse'][best_fold_index_r2] 
    
for fold_index in range(cv):
    r2_fold = search2.cv_results_['split{}_test_r2'.format(fold_index)][best_fold_index_r2]
    rmse_fold = search2.cv_results_['split{}_test_rmse'.format(fold_index)][best_fold_index_r2]

    print(f"Fold {fold_index + 1} - R^2: {r2_fold}, RMSE: {rmse_fold}")

    test_indices_fold = test_indices[fold_index]

    print(f"Fold {fold_index + 1} - Test Indices: {test_indices_fold}", '\n')

print('\n', 'Average R2 score:', average_r2)
print('Average RMSE:', average_rmse)

best_gp2 = search2.best_estimator_

## GP for Final discharge capacity (100th)

In [ ]:
kernel3 = Matern()

param_dist3 = {'alpha': np.logspace(-10,0,11),
               'kernel__length_scale': np.logspace(-5,5,11),
               'kernel__nu': np.linspace(0.5,10,20)
              }


model_gp3 = GaussianProcessRegressor(kernel= kernel3,
                                 alpha = 'alpha',
                                 optimizer=None,
                                 random_state=0)

search3 = RandomizedSearchCV(model_gp3, param_distributions=param_dist3,
                             n_iter = n_iter, cv = cv, scoring={'r2': custom_r2_scorer,'rmse': custom_rmse_scorer},  
                             n_jobs=-1, verbose=1, refit='r2')

search3.fit(x_train, y_train3_std)

print(search3.best_params_, '\n')

best_fold_index_r2 = np.argmax(search3.cv_results_['mean_test_r2'])

kf = KFold(n_splits=cv)
test_indices = []

for _, test_index in kf.split(x_train):
    test_indices.append(test_index)

average_r2 = search3.cv_results_['mean_test_r2'][best_fold_index_r2]
average_rmse = search3.cv_results_['mean_test_rmse'][best_fold_index_r2] 
    
for fold_index in range(cv):
    r2_fold = search3.cv_results_['split{}_test_r2'.format(fold_index)][best_fold_index_r2]
    rmse_fold = search3.cv_results_['split{}_test_rmse'.format(fold_index)][best_fold_index_r2]

    print(f"Fold {fold_index + 1} - R^2: {r2_fold}, RMSE: {rmse_fold}")

    test_indices_fold = test_indices[fold_index]

    print(f"Fold {fold_index + 1} - Test Indices: {test_indices_fold}", '\n')

print('\n', 'Average R2 score:', average_r2)
print('Average RMSE:', average_rmse)

best_gp3 = search3.best_estimator_

## GP for Latter retention (76~100 cycle)

In [ ]:
kernel4 = Matern()

param_dist4 = {'alpha': np.logspace(-10,0,11),
               'kernel__length_scale': np.logspace(-5,5,11),
               'kernel__nu': np.linspace(0.5,10,20)
              }


model_gp4 = GaussianProcessRegressor(kernel= kernel4,
                                 alpha = 'alpha',
                                 optimizer=None,
                                 random_state=0)

search4 = RandomizedSearchCV(model_gp4, param_distributions=param_dist4,
                             n_iter = n_iter, cv = cv, scoring={'r2': custom_r2_scorer,'rmse': custom_rmse_scorer},  
                             n_jobs=-1, verbose=1, refit='r2')

search4.fit(x_train, y_train4_std)

print(search4.best_params_, '\n')

best_fold_index_r2 = np.argmax(search4.cv_results_['mean_test_r2'])

kf = KFold(n_splits=cv)
test_indices = []

for _, test_index in kf.split(x_train):
    test_indices.append(test_index)

average_r2 = search4.cv_results_['mean_test_r2'][best_fold_index_r2]
average_rmse = search4.cv_results_['mean_test_rmse'][best_fold_index_r2] 
    
for fold_index in range(cv):
    r2_fold = search4.cv_results_['split{}_test_r2'.format(fold_index)][best_fold_index_r2]
    rmse_fold = search4.cv_results_['split{}_test_rmse'.format(fold_index)][best_fold_index_r2]

    print(f"Fold {fold_index + 1} - R^2: {r2_fold}, RMSE: {rmse_fold}")

    test_indices_fold = test_indices[fold_index]

    print(f"Fold {fold_index + 1} - Test Indices: {test_indices_fold}", '\n')

print('\n', 'Average R2 score:', average_r2)
print('Average RMSE:', average_rmse)

best_gp4 = search4.best_estimator_

# Find out the next experiment point with constraied EI

In [ ]:
warnings.filterwarnings('ignore')

best_mu1 = np.nanmin(y_train1_std)
best_mu1_inv = best_mu1 * std(y_train1, axis=0) + mean(y_train1, axis=0)
best_mu2 = np.nanmax(y_train2_std)
best_mu2_inv = best_mu2 * std(y_train2, axis=0) + mean(y_train2, axis=0)
best_mu3 = np.nanmax(y_train3_std)
best_mu3_inv = best_mu3 * std(y_train3, axis=0) + mean(y_train3, axis=0)
best_mu4 = np.nanmax(y_train4_std)
best_mu4_inv = best_mu4 * std(y_train4, axis=0) + mean(y_train4, axis=0)

def EI(a, best_gp1, best_gp2, best_gp3, best_gp4,
       best_mu1, best_mu2, best_mu3, best_mu4, e= 0): 
    
    X_new = search_space_final[int(a),:]
    X_new = X_new.reshape(1,-1)
    
    mu1, sigma1 = best_gp1.predict(X_new, return_std = True)
    z1 = np.zeros(mu1.shape)
    z11 = np.zeros(mu1.shape)
    z1[sigma1 > 0 ] = ((best_mu1 - mu1 - e)/sigma1)[sigma1 > 0]
    const1 = ((6 - mean(y_train1, axis=0)) / std(y_train1, axis=0))
    z11[sigma1 > 0 ] = ((mu1 - const1)/sigma1)[sigma1 > 0]
    EI1 = ((best_mu1 - mu1 - e) * norm.cdf(z1) + sigma1 * norm.pdf(z1))
    EI1_new = EI1 * (1 - norm.cdf(z11))
    
    
    mu2, sigma2 = best_gp2.predict(X_new, return_std = True)
    z2 = np.zeros(mu2.shape)
    z22 = np.zeros(mu2.shape)
    z2[sigma2 > 0 ] = ((mu2 - best_mu2 - e)/sigma2)[sigma2 > 0]
    const2 = ((80 - mean(y_train2, axis=0)) / std(y_train2, axis=0))           
    z22[sigma2 > 0 ] = ((mu2 - const2)/sigma2)[sigma2 > 0]
    EI2 = ((mu2 - best_mu2- e) * norm.cdf(z2) + sigma2 * norm.pdf(z2))
    EI2_new = EI2 * (norm.cdf(z22)) 
    
    
    mu3, sigma3 = best_gp3.predict(X_new, return_std = True)
    z3 = np.zeros(mu3.shape)
    z33 = np.zeros(mu3.shape)
    z3[sigma3 > 0 ] = ((mu3 - best_mu3 - e)/sigma3)[sigma3 > 0]
    const3 = ((148 - mean(y_train3, axis=0)) / std(y_train3, axis=0))           
    z33[sigma3 > 0 ] = ((mu3 - const3)/sigma3)[sigma3 > 0]
    EI3 = ((mu3 - best_mu3- e) * norm.cdf(z3) + sigma3 * norm.pdf(z3))
    EI3_new = EI3 * (norm.cdf(z33))
    
    
    mu4, sigma4 = best_gp4.predict(X_new, return_std = True)
    z4 = np.zeros(mu4.shape)
    z44 = np.zeros(mu4.shape)
    z4[sigma4 > 0 ] = ((mu4 - best_mu4 - e)/sigma4)[sigma4 > 0]
    const4 = ((90 - mean(y_train4, axis=0)) / std(y_train4, axis=0))           
    z44[sigma4 > 0 ] = ((mu4 - const4)/sigma4)[sigma4 > 0]
    EI4 = ((mu4 - best_mu4- e) * norm.cdf(z4) + sigma4 * norm.pdf(z4))
    EI4_new = EI4 * (norm.cdf(z44))

    
    return (EI1_new + 30*EI2_new + 20*EI3_new + 15*EI4_new)

EI_values = np.empty((search_space_final.shape[0],1))

for i in range(0, search_space_final.shape[0]):
    EI_value = EI(i, best_gp1, best_gp2, best_gp3, best_gp4,
                  best_mu1, best_mu2, best_mu3, best_mu4, e= 0)
    EI_values[i,0] = EI_value

result_new = np.nanmax(EI_values)
result_new_index = np.where(EI_values == result_new)[0]
print('---------------------------------------------------')
print('Best SET:', best_mu1_inv.tolist())
print('Best Retention:', best_mu2_inv.tolist())
print('Best Final discharge capacity (100th):', best_mu3_inv.tolist())
print('Best Latter retention (76~100 cycle):', best_mu4_inv.tolist(), '\n')

print("Objective function", result_new)
print("EI vlaue :", EI(result_new_index, best_gp1, best_gp2, best_gp3, best_gp4,
                  best_mu1, best_mu2, best_mu3, best_mu4, e= 0))
print('Next experiment point:', search_space_final[result_new_index, :])
pred1 = best_gp1.predict(search_space_final[result_new_index, :].reshape(1,-1))
pred1_inv = pred1 * std(y_train1, axis=0) + mean(y_train1, axis=0)
print('SET:', pred1_inv.tolist())
pred2 = best_gp2.predict(search_space_final[result_new_index, :].reshape(1,-1))
pred2_inv = pred2 * std(y_train2, axis=0) + mean(y_train2, axis=0)
print('Retention:', pred2_inv.tolist())
pred3 = best_gp3.predict(search_space_final[result_new_index, :].reshape(1,-1))
pred3_inv = pred3 * std(y_train3, axis=0) + mean(y_train3, axis=0)
print('Final discharge capacity (100th):', pred3_inv.tolist())
pred4 = best_gp4.predict(search_space_final[result_new_index, :].reshape(1,-1))
pred4_inv = pred4 * std(y_train4, axis=0) + mean(y_train4, axis=0)
print('Latter retention (76~100 cycle):', pred4_inv.tolist())
print('---------------------------------------------------\n')

print(EI(result_new_index, best_gp1, best_gp2, best_gp3, best_gp4,
         best_mu1, best_mu2, best_mu3, best_mu4, e= 0))

search_space_final = np.delete(search_space_final, result_new_index, axis = 0)
search_space_final.shape

# Gernerate additional experiment points for parallel experiment

In [ ]:
EI_values = np.empty((search_space_final.shape[0],1))

for i in range(0, search_space_final.shape[0]):
    EI_value = EI(i, best_gp1, best_gp2, best_gp3, best_gp4,
                  best_mu1, best_mu2, best_mu3, best_mu4, e= 0)
    EI_values[i,0] = EI_value


#-----------------------------------------------------------------------------------------------------------------

explore_level = 0.05
index_relevant = np.argpartition(EI_values, -int(explore_level * EI_values.shape[0]), axis=0)[::-1]
partitioned_area = search_space_final[index_relevant,:]
Top_1_EI = partitioned_area[0:int(explore_level * EI_values.shape[0])+1,:]

#-----------------------------------------------------------------------------------------------------------------
Std_values = np.empty((int(explore_level * EI_values.shape[0]),1))

for i in range(0, int(explore_level * EI_values.shape[0])):
    mu_1, std_1 = best_gp1.predict(Top_1_EI[i,:], return_std = True)
    mu_2, std_2 = best_gp2.predict(Top_1_EI[i,:], return_std = True)
    mu_3, std_3 = best_gp3.predict(Top_1_EI[i,:], return_std = True)
    mu_4, std_4 = best_gp4.predict(Top_1_EI[i,:], return_std = True)
    Total_std = (std_1**2 + std_2**2 + std_3**2 + std_4**2)**0.5
    Std_values[i,0] = Total_std

Next_point_index = Std_values.argsort(axis=0)[::-1]

add_num = 2 
Next_point_index = Next_point_index[:add_num,:]
Next_point = Top_1_EI[Next_point_index,:]
Next_point = np.squeeze(Next_point)

for i in range (0,add_num):
    print('Additional experiment points ', i+1, ' \n\n', Next_point[i,:], '\n')
    next1 = best_gp1.predict(Next_point[i,:].reshape(1,-1))
    next1_inv = next1 * std(y_train1, axis=0) + mean(y_train1, axis=0)
    print('SET:', next1_inv.tolist())
    next2 = best_gp2.predict(Next_point[i,:].reshape(1,-1))
    next2_inv = next2 * std(y_train2, axis=0) + mean(y_train2, axis=0)
    print('Retention:', next2_inv.tolist())
    next3 = best_gp3.predict(Next_point[i,:].reshape(1,-1))
    next3_inv = next3 * std(y_train3, axis=0) + mean(y_train3, axis=0)
    print('Final discharge capacity (100th):', next3_inv.tolist())
    next4 = best_gp4.predict(Next_point[i,:].reshape(1,-1))
    next4_inv = next4 * std(y_train4, axis=0) + mean(y_train4, axis=0)
    print('Latter retention (76~100 cycle):', next4_inv.tolist(),'\n\n')
  
    matching_rows = np.where((search_space_final == Next_point[i,:]).all(axis=1))
    mu_1, std_1 = best_gp1.predict(Next_point[i,:].reshape(1,-1), return_std = True)
    mu_2, std_2 = best_gp2.predict(Next_point[i,:].reshape(1,-1), return_std = True)
    mu_3, std_3 = best_gp3.predict(Next_point[i,:].reshape(1,-1), return_std = True)
    mu_4, std_4 = best_gp4.predict(Next_point[i,:].reshape(1,-1), return_std = True)
    Total_std = (std_1**2 + std_2**2 + std_3**2 + std_4**2)**0.5
    search_space_final = np.delete(search_space_final, matching_rows, axis = 0)
    print('-----------------------------------------------\n\n')
    
    
search_space_final.shape

# New experiment data import

In [ ]:
exp_data_new = pd.read_csv(r"") #The experimental results for the 1st candidates proposed by AL-DoE
display(exp_data_new)

print("\n")

exp_data_x_new = exp_data_new.iloc[:, [0,1,2]]
exp_data_y_new = exp_data_new.iloc[:, [3,4,5,6]]
display(exp_data_x_new)
display(exp_data_y_new)

print("\n")

x_train_new = exp_data_x_new
y_train1_new = exp_data_y_new.iloc[:, [0]]
y_train2_new = exp_data_y_new.iloc[:, [1]]
y_train3_new = exp_data_y_new.iloc[:, [2]]
y_train4_new = exp_data_y_new.iloc[:, [3]]

display(x_train_new)
display(y_train1_new)
display(y_train2_new)
display(y_train3_new)
display(y_train4_new)

In [ ]:
x_train = pd.concat([x_train, x_train_new])
x_train = x_train.sample(frac=1, random_state=2023).reset_index(drop=True)
y_train1 = pd.concat([y_train1, y_train1_new])
y_train1 = y_train1.sample(frac=1, random_state=2023).reset_index(drop=True)
y_train2 = pd.concat([y_train2, y_train2_new])
y_train2 = y_train2.sample(frac=1, random_state=2023).reset_index(drop=True)
y_train3 = pd.concat([y_train3, y_train3_new])
y_train3 = y_train3.sample(frac=1, random_state=2023).reset_index(drop=True)
y_train4 = pd.concat([y_train4, y_train4_new])
y_train4 = y_train4.sample(frac=1, random_state=2023).reset_index(drop=True)

display(x_train)
display(y_train1)
display(y_train2)
display(y_train3)
display(y_train4)

y_train1_std = (y_train1 - mean(y_train1, axis=0)) / std(y_train1, axis=0)
y_train2_std = (y_train2 - mean(y_train2, axis=0)) / std(y_train2, axis=0)
y_train3_std = (y_train3 - mean(y_train3, axis=0)) / std(y_train3, axis=0)
y_train4_std = (y_train4 - mean(y_train4, axis=0)) / std(y_train4, axis=0)

print(y_train1_std)
print(y_train2_std)
print(y_train3_std)
print(y_train4_std)

# Retrain and retest GP model with hyperparameter tuning (2nd iteration)

## GP for SET

In [ ]:
kernel1 = Matern()

param_dist1 = {'alpha': np.logspace(-10,0,11),
               'kernel__length_scale': np.logspace(-5,5,11),
               'kernel__nu': np.linspace(0.5,10,20)
              }

model_gp1 = GaussianProcessRegressor(kernel= kernel1,
                                 alpha = 'alpha',
                                 optimizer=None,
                                 random_state=0)

search1 = RandomizedSearchCV(model_gp1, param_distributions=param_dist1,
                             n_iter = n_iter, cv = cv, scoring={'r2': custom_r2_scorer,'rmse': custom_rmse_scorer},  
                             n_jobs=-1, verbose=1, refit='r2')

search1.fit(x_train, y_train1_std)

print(search1.best_params_, '\n')

best_fold_index_r2 = np.argmax(search1.cv_results_['mean_test_r2'])

kf = KFold(n_splits=cv)
test_indices = []

for _, test_index in kf.split(x_train):
    test_indices.append(test_index)

average_r2 = search1.cv_results_['mean_test_r2'][best_fold_index_r2]
average_rmse = search1.cv_results_['mean_test_rmse'][best_fold_index_r2] 
    
for fold_index in range(cv):
    r2_fold = search1.cv_results_['split{}_test_r2'.format(fold_index)][best_fold_index_r2]
    rmse_fold = search1.cv_results_['split{}_test_rmse'.format(fold_index)][best_fold_index_r2]

    print(f"Fold {fold_index + 1} - R^2: {r2_fold}, RMSE: {rmse_fold}")

    test_indices_fold = test_indices[fold_index]

    print(f"Fold {fold_index + 1} - Test Indices: {test_indices_fold}", '\n')

print('\n', 'Average R2 score:', average_r2)
print('Average RMSE:', average_rmse)

best_gp1 = search1.best_estimator_

## GP for retention

In [ ]:
kernel2 = Matern()

param_dist2 = {'alpha': np.logspace(-10,0,11),
               'kernel__length_scale': np.logspace(-5,5,11),
               'kernel__nu': np.linspace(0.5,10,20)
              }


model_gp2 = GaussianProcessRegressor(kernel= kernel2,
                                 alpha = 'alpha',
                                 optimizer=None,
                                 random_state=0)

search2 = RandomizedSearchCV(model_gp2, param_distributions=param_dist2,
                             n_iter = n_iter, cv = cv, scoring={'r2': custom_r2_scorer,'rmse': custom_rmse_scorer},  
                             n_jobs=-1, verbose=1, refit='r2')

search2.fit(x_train, y_train2_std)

print(search2.best_params_, '\n')

best_fold_index_r2 = np.argmax(search2.cv_results_['mean_test_r2'])

kf = KFold(n_splits=cv)
test_indices = []

for _, test_index in kf.split(x_train):
    test_indices.append(test_index)

average_r2 = search2.cv_results_['mean_test_r2'][best_fold_index_r2]
average_rmse = search2.cv_results_['mean_test_rmse'][best_fold_index_r2] 
    
for fold_index in range(cv):
    r2_fold = search2.cv_results_['split{}_test_r2'.format(fold_index)][best_fold_index_r2]
    rmse_fold = search2.cv_results_['split{}_test_rmse'.format(fold_index)][best_fold_index_r2]

    print(f"Fold {fold_index + 1} - R^2: {r2_fold}, RMSE: {rmse_fold}")

    test_indices_fold = test_indices[fold_index]

    print(f"Fold {fold_index + 1} - Test Indices: {test_indices_fold}", '\n')

print('\n', 'Average R2 score:', average_r2)
print('Average RMSE:', average_rmse)

best_gp2 = search2.best_estimator_

## GP for Final discharge capacity (100th)

In [ ]:
kernel3 = Matern()

param_dist3 = {'alpha': np.logspace(-10,0,11),
               'kernel__length_scale': np.logspace(-5,5,11),
               'kernel__nu': np.linspace(0.5,10,20)
              }


model_gp3 = GaussianProcessRegressor(kernel= kernel3,
                                 alpha = 'alpha',
                                 optimizer=None,
                                 random_state=0)

search3 = RandomizedSearchCV(model_gp3, param_distributions=param_dist3,
                             n_iter = n_iter, cv = cv, scoring={'r2': custom_r2_scorer,'rmse': custom_rmse_scorer},  
                             n_jobs=-1, verbose=1, refit='r2')

search3.fit(x_train, y_train3_std)

print(search3.best_params_, '\n')

best_fold_index_r2 = np.argmax(search3.cv_results_['mean_test_r2'])

kf = KFold(n_splits=cv)
test_indices = []

for _, test_index in kf.split(x_train):
    test_indices.append(test_index)

average_r2 = search3.cv_results_['mean_test_r2'][best_fold_index_r2]
average_rmse = search3.cv_results_['mean_test_rmse'][best_fold_index_r2] 
    
for fold_index in range(cv):
    r2_fold = search3.cv_results_['split{}_test_r2'.format(fold_index)][best_fold_index_r2]
    rmse_fold = search3.cv_results_['split{}_test_rmse'.format(fold_index)][best_fold_index_r2]

    print(f"Fold {fold_index + 1} - R^2: {r2_fold}, RMSE: {rmse_fold}")

    test_indices_fold = test_indices[fold_index]

    print(f"Fold {fold_index + 1} - Test Indices: {test_indices_fold}", '\n')

print('\n', 'Average R2 score:', average_r2)
print('Average RMSE:', average_rmse)

best_gp3 = search3.best_estimator_

## GP for Latter retention (76~100 cycle)

In [ ]:
kernel4 = Matern()

param_dist4 = {'alpha': np.logspace(-10,0,11),
               'kernel__length_scale': np.logspace(-5,5,11),
               'kernel__nu': np.linspace(0.5,10,20)
              }


model_gp4 = GaussianProcessRegressor(kernel= kernel4,
                                 alpha = 'alpha',
                                 optimizer=None,
                                 random_state=0)

search4 = RandomizedSearchCV(model_gp4, param_distributions=param_dist4,
                             n_iter = n_iter, cv = cv, scoring={'r2': custom_r2_scorer,'rmse': custom_rmse_scorer},  
                             n_jobs=-1, verbose=1, refit='r2')

search4.fit(x_train, y_train4_std)

print(search4.best_params_, '\n')

best_fold_index_r2 = np.argmax(search4.cv_results_['mean_test_r2'])

kf = KFold(n_splits=cv)
test_indices = []

for _, test_index in kf.split(x_train):
    test_indices.append(test_index)

average_r2 = search4.cv_results_['mean_test_r2'][best_fold_index_r2]
average_rmse = search4.cv_results_['mean_test_rmse'][best_fold_index_r2] 
    
for fold_index in range(cv):
    r2_fold = search4.cv_results_['split{}_test_r2'.format(fold_index)][best_fold_index_r2]
    rmse_fold = search4.cv_results_['split{}_test_rmse'.format(fold_index)][best_fold_index_r2]

    print(f"Fold {fold_index + 1} - R^2: {r2_fold}, RMSE: {rmse_fold}")

    test_indices_fold = test_indices[fold_index]

    print(f"Fold {fold_index + 1} - Test Indices: {test_indices_fold}", '\n')

print('\n', 'Average R2 score:', average_r2)
print('Average RMSE:', average_rmse)

best_gp4 = search4.best_estimator_

# Find out the next experiment point with constraied EI

In [ ]:
warnings.filterwarnings('ignore')

best_mu1 = np.nanmin(y_train1_std)
best_mu1_inv = best_mu1 * std(y_train1, axis=0) + mean(y_train1, axis=0)
best_mu2 = np.nanmax(y_train2_std)
best_mu2_inv = best_mu2 * std(y_train2, axis=0) + mean(y_train2, axis=0)
best_mu3 = np.nanmax(y_train3_std)
best_mu3_inv = best_mu3 * std(y_train3, axis=0) + mean(y_train3, axis=0)
best_mu4 = np.nanmax(y_train4_std)
best_mu4_inv = best_mu4 * std(y_train4, axis=0) + mean(y_train4, axis=0)

def EI(a, best_gp1, best_gp2, best_gp3, best_gp4,
       best_mu1, best_mu2, best_mu3, best_mu4, e= 0): 
    
    X_new = search_space_final[int(a),:]
    X_new = X_new.reshape(1,-1)
    
    mu1, sigma1 = best_gp1.predict(X_new, return_std = True)
    z1 = np.zeros(mu1.shape)
    z11 = np.zeros(mu1.shape)
    z1[sigma1 > 0 ] = ((best_mu1 - mu1 - e)/sigma1)[sigma1 > 0]
    const1 = ((6 - mean(y_train1, axis=0)) / std(y_train1, axis=0))
    z11[sigma1 > 0 ] = ((mu1 - const1)/sigma1)[sigma1 > 0]
    EI1 = ((best_mu1 - mu1 - e) * norm.cdf(z1) + sigma1 * norm.pdf(z1))
    EI1_new = EI1 * (1 - norm.cdf(z11))
    
    
    mu2, sigma2 = best_gp2.predict(X_new, return_std = True)
    z2 = np.zeros(mu2.shape)
    z22 = np.zeros(mu2.shape)
    z2[sigma2 > 0 ] = ((mu2 - best_mu2 - e)/sigma2)[sigma2 > 0]
    const2 = ((80 - mean(y_train2, axis=0)) / std(y_train2, axis=0))           
    z22[sigma2 > 0 ] = ((mu2 - const2)/sigma2)[sigma2 > 0]
    EI2 = ((mu2 - best_mu2- e) * norm.cdf(z2) + sigma2 * norm.pdf(z2))
    EI2_new = EI2 * (norm.cdf(z22)) 
    
    
    mu3, sigma3 = best_gp3.predict(X_new, return_std = True)
    z3 = np.zeros(mu3.shape)
    z33 = np.zeros(mu3.shape)
    z3[sigma3 > 0 ] = ((mu3 - best_mu3 - e)/sigma3)[sigma3 > 0]
    const3 = ((148 - mean(y_train3, axis=0)) / std(y_train3, axis=0))           
    z33[sigma3 > 0 ] = ((mu3 - const3)/sigma3)[sigma3 > 0]
    EI3 = ((mu3 - best_mu3- e) * norm.cdf(z3) + sigma3 * norm.pdf(z3))
    EI3_new = EI3 * (norm.cdf(z33))
    
    
    mu4, sigma4 = best_gp4.predict(X_new, return_std = True)
    z4 = np.zeros(mu4.shape)
    z44 = np.zeros(mu4.shape)
    z4[sigma4 > 0 ] = ((mu4 - best_mu4 - e)/sigma4)[sigma4 > 0]
    const4 = ((90 - mean(y_train4, axis=0)) / std(y_train4, axis=0))           
    z44[sigma4 > 0 ] = ((mu4 - const4)/sigma4)[sigma4 > 0]
    EI4 = ((mu4 - best_mu4- e) * norm.cdf(z4) + sigma4 * norm.pdf(z4))
    EI4_new = EI4 * (norm.cdf(z44))

    
    return (EI1_new + 30*EI2_new + 20*EI3_new + 15*EI4_new)

EI_values = np.empty((search_space_final.shape[0],1))

for i in range(0, search_space_final.shape[0]):
    EI_value = EI(i, best_gp1, best_gp2, best_gp3, best_gp4,
                  best_mu1, best_mu2, best_mu3, best_mu4, e= 0)
    EI_values[i,0] = EI_value

result_new = np.nanmax(EI_values)
result_new_index = np.where(EI_values == result_new)[0]
print('---------------------------------------------------')
print('Best SET:', best_mu1_inv.tolist())
print('Best Retention:', best_mu2_inv.tolist())
print('Best Final discharge capacity (100th):', best_mu3_inv.tolist())
print('Best Latter retention (76~100 cycle):', best_mu4_inv.tolist(), '\n')

print("Objective function", result_new)
print("EI vlaue :", EI(result_new_index, best_gp1, best_gp2, best_gp3, best_gp4,
                  best_mu1, best_mu2, best_mu3, best_mu4, e= 0))
print('Next experiment point:', search_space_final[result_new_index, :])
pred1 = best_gp1.predict(search_space_final[result_new_index, :].reshape(1,-1))
pred1_inv = pred1 * std(y_train1, axis=0) + mean(y_train1, axis=0)
print('SET:', pred1_inv.tolist())
pred2 = best_gp2.predict(search_space_final[result_new_index, :].reshape(1,-1))
pred2_inv = pred2 * std(y_train2, axis=0) + mean(y_train2, axis=0)
print('Retention:', pred2_inv.tolist())
pred3 = best_gp3.predict(search_space_final[result_new_index, :].reshape(1,-1))
pred3_inv = pred3 * std(y_train3, axis=0) + mean(y_train3, axis=0)
print('Final discharge capacity (100th):', pred3_inv.tolist())
pred4 = best_gp4.predict(search_space_final[result_new_index, :].reshape(1,-1))
pred4_inv = pred4 * std(y_train4, axis=0) + mean(y_train4, axis=0)
print('Latter retention (76~100 cycle):', pred4_inv.tolist())
print('---------------------------------------------------\n')

print(EI(result_new_index, best_gp1, best_gp2, best_gp3, best_gp4,
         best_mu1, best_mu2, best_mu3, best_mu4, e= 0))

search_space_final = np.delete(search_space_final, result_new_index, axis = 0)
search_space_final.shape

# Gernerate additional experiment points for parallel experiment

In [ ]:
EI_values = np.empty((search_space_final.shape[0],1))

for i in range(0, search_space_final.shape[0]):
    EI_value = EI(i, best_gp1, best_gp2, best_gp3, best_gp4,
                  best_mu1, best_mu2, best_mu3, best_mu4, e= 0)
    EI_values[i,0] = EI_value


#-----------------------------------------------------------------------------------------------------------------

explore_level = 0.05
index_relevant = np.argpartition(EI_values, -int(explore_level * EI_values.shape[0]), axis=0)[::-1]
partitioned_area = search_space_final[index_relevant,:]
Top_1_EI = partitioned_area[0:int(explore_level * EI_values.shape[0])+1,:]

#-----------------------------------------------------------------------------------------------------------------
Std_values = np.empty((int(explore_level * EI_values.shape[0]),1))

for i in range(0, int(explore_level * EI_values.shape[0])):
    mu_1, std_1 = best_gp1.predict(Top_1_EI[i,:], return_std = True)
    mu_2, std_2 = best_gp2.predict(Top_1_EI[i,:], return_std = True)
    mu_3, std_3 = best_gp3.predict(Top_1_EI[i,:], return_std = True)
    mu_4, std_4 = best_gp4.predict(Top_1_EI[i,:], return_std = True)
    Total_std = (std_1**2 + std_2**2 + std_3**2 + std_4**2)**0.5
    Std_values[i,0] = Total_std

Next_point_index = Std_values.argsort(axis=0)[::-1]

add_num = 2 
Next_point_index = Next_point_index[:add_num,:]
Next_point = Top_1_EI[Next_point_index,:]
Next_point = np.squeeze(Next_point)

for i in range (0,add_num):
    print('Additional experiment points ', i+1, ' \n\n', Next_point[i,:], '\n')
    next1 = best_gp1.predict(Next_point[i,:].reshape(1,-1))
    next1_inv = next1 * std(y_train1, axis=0) + mean(y_train1, axis=0)
    print('SET:', next1_inv.tolist())
    next2 = best_gp2.predict(Next_point[i,:].reshape(1,-1))
    next2_inv = next2 * std(y_train2, axis=0) + mean(y_train2, axis=0)
    print('Retention:', next2_inv.tolist())
    next3 = best_gp3.predict(Next_point[i,:].reshape(1,-1))
    next3_inv = next3 * std(y_train3, axis=0) + mean(y_train3, axis=0)
    print('Final discharge capacity (100th):', next3_inv.tolist())
    next4 = best_gp4.predict(Next_point[i,:].reshape(1,-1))
    next4_inv = next4 * std(y_train4, axis=0) + mean(y_train4, axis=0)
    print('Latter retention (76~100 cycle):', next4_inv.tolist(),'\n\n')
  
    matching_rows = np.where((search_space_final == Next_point[i,:]).all(axis=1))
    mu_1, std_1 = best_gp1.predict(Next_point[i,:].reshape(1,-1), return_std = True)
    mu_2, std_2 = best_gp2.predict(Next_point[i,:].reshape(1,-1), return_std = True)
    mu_3, std_3 = best_gp3.predict(Next_point[i,:].reshape(1,-1), return_std = True)
    mu_4, std_4 = best_gp4.predict(Next_point[i,:].reshape(1,-1), return_std = True)
    Total_std = (std_1**2 + std_2**2 + std_3**2 + std_4**2)**0.5
    search_space_final = np.delete(search_space_final, matching_rows, axis = 0)
    print('-----------------------------------------------\n\n')
    
    
search_space_final.shape

# New experiment data import

In [ ]:
exp_data_new = pd.read_csv(r"") #The experimental results for the 2nd candidates proposed by AL-DoE
display(exp_data_new)

print("\n")

exp_data_x_new = exp_data_new.iloc[:, [0,1,2]] 
exp_data_y_new = exp_data_new.iloc[:, [3,4,5,6]]
display(exp_data_x_new)
display(exp_data_y_new)

print("\n")

x_train_new = exp_data_x_new
y_train1_new = exp_data_y_new.iloc[:, [0]]
y_train2_new = exp_data_y_new.iloc[:, [1]]
y_train3_new = exp_data_y_new.iloc[:, [2]]
y_train4_new = exp_data_y_new.iloc[:, [3]]

display(x_train_new)
display(y_train1_new)
display(y_train2_new)
display(y_train3_new)
display(y_train4_new)

In [ ]:
x_train = pd.concat([x_train, x_train_new])
x_train = x_train.sample(frac=1, random_state=2024).reset_index(drop=True)
y_train1 = pd.concat([y_train1, y_train1_new])
y_train1 = y_train1.sample(frac=1, random_state=2024).reset_index(drop=True)
y_train2 = pd.concat([y_train2, y_train2_new])
y_train2 = y_train2.sample(frac=1, random_state=2024).reset_index(drop=True)
y_train3 = pd.concat([y_train3, y_train3_new])
y_train3 = y_train3.sample(frac=1, random_state=2024).reset_index(drop=True)
y_train4 = pd.concat([y_train4, y_train4_new])
y_train4 = y_train4.sample(frac=1, random_state=2024).reset_index(drop=True)

display(x_train)
display(y_train1)
display(y_train2)
display(y_train3)
display(y_train4)

y_train1_std = (y_train1 - mean(y_train1, axis=0)) / std(y_train1, axis=0)
y_train2_std = (y_train2 - mean(y_train2, axis=0)) / std(y_train2, axis=0)
y_train3_std = (y_train3 - mean(y_train3, axis=0)) / std(y_train3, axis=0)
y_train4_std = (y_train4 - mean(y_train4, axis=0)) / std(y_train4, axis=0)

print(y_train1_std)
print(y_train2_std)
print(y_train3_std)
print(y_train4_std)

# Retrain and retest GP model with hyperparameter tuning (3rd iteration)

## GP for SET

In [ ]:
kernel1 = Matern()

param_dist1 = {'alpha': np.logspace(-10,0,11),
               'kernel__length_scale': np.logspace(-5,5,11),
               'kernel__nu': np.linspace(0.5,10,20)
              }

model_gp1 = GaussianProcessRegressor(kernel= kernel1,
                                 alpha = 'alpha',
                                 optimizer=None,
                                 random_state=0)

search1 = RandomizedSearchCV(model_gp1, param_distributions=param_dist1,
                             n_iter = n_iter, cv = cv, scoring={'r2': custom_r2_scorer,'rmse': custom_rmse_scorer},  
                             n_jobs=-1, verbose=1, refit='r2')

search1.fit(x_train, y_train1_std)

print(search1.best_params_, '\n')

best_fold_index_r2 = np.argmax(search1.cv_results_['mean_test_r2'])

kf = KFold(n_splits=cv)
test_indices = []

for _, test_index in kf.split(x_train):
    test_indices.append(test_index)

average_r2 = search1.cv_results_['mean_test_r2'][best_fold_index_r2]
average_rmse = search1.cv_results_['mean_test_rmse'][best_fold_index_r2] 
    
for fold_index in range(cv):
    r2_fold = search1.cv_results_['split{}_test_r2'.format(fold_index)][best_fold_index_r2]
    rmse_fold = search1.cv_results_['split{}_test_rmse'.format(fold_index)][best_fold_index_r2]

    print(f"Fold {fold_index + 1} - R^2: {r2_fold}, RMSE: {rmse_fold}")

    test_indices_fold = test_indices[fold_index]

    print(f"Fold {fold_index + 1} - Test Indices: {test_indices_fold}", '\n')

print('\n', 'Average R2 score:', average_r2)
print('Average RMSE:', average_rmse)

best_gp1 = search1.best_estimator_

## GP for retention

In [ ]:
kernel2 = Matern()

param_dist2 = {'alpha': np.logspace(-10,0,11),
               'kernel__length_scale': np.logspace(-5,5,11),
               'kernel__nu': np.linspace(0.5,10,20)
              }


model_gp2 = GaussianProcessRegressor(kernel= kernel2,
                                 alpha = 'alpha',
                                 optimizer=None,
                                 random_state=0)

search2 = RandomizedSearchCV(model_gp2, param_distributions=param_dist2,
                             n_iter = n_iter, cv = cv, scoring={'r2': custom_r2_scorer,'rmse': custom_rmse_scorer},  
                             n_jobs=-1, verbose=1, refit='r2')

search2.fit(x_train, y_train2_std)

print(search2.best_params_, '\n')

best_fold_index_r2 = np.argmax(search2.cv_results_['mean_test_r2'])

kf = KFold(n_splits=cv)
test_indices = []

for _, test_index in kf.split(x_train):
    test_indices.append(test_index)

average_r2 = search2.cv_results_['mean_test_r2'][best_fold_index_r2]
average_rmse = search2.cv_results_['mean_test_rmse'][best_fold_index_r2] 
    
for fold_index in range(cv):
    r2_fold = search2.cv_results_['split{}_test_r2'.format(fold_index)][best_fold_index_r2]
    rmse_fold = search2.cv_results_['split{}_test_rmse'.format(fold_index)][best_fold_index_r2]

    print(f"Fold {fold_index + 1} - R^2: {r2_fold}, RMSE: {rmse_fold}")

    test_indices_fold = test_indices[fold_index]

    print(f"Fold {fold_index + 1} - Test Indices: {test_indices_fold}", '\n')

print('\n', 'Average R2 score:', average_r2)
print('Average RMSE:', average_rmse)

best_gp2 = search2.best_estimator_

## GP for Final discharge capacity (100th)

In [ ]:
kernel3 = Matern()

param_dist3 = {'alpha': np.logspace(-10,0,11),
               'kernel__length_scale': np.logspace(-5,5,11),
               'kernel__nu': np.linspace(0.5,10,20)
              }


model_gp3 = GaussianProcessRegressor(kernel= kernel3,
                                 alpha = 'alpha',
                                 optimizer=None,
                                 random_state=0)

search3 = RandomizedSearchCV(model_gp3, param_distributions=param_dist3,
                             n_iter = n_iter, cv = cv, scoring={'r2': custom_r2_scorer,'rmse': custom_rmse_scorer},  
                             n_jobs=-1, verbose=1, refit='r2')

search3.fit(x_train, y_train3_std)

print(search3.best_params_, '\n')

best_fold_index_r2 = np.argmax(search3.cv_results_['mean_test_r2'])

kf = KFold(n_splits=cv)
test_indices = []

for _, test_index in kf.split(x_train):
    test_indices.append(test_index)

average_r2 = search3.cv_results_['mean_test_r2'][best_fold_index_r2]
average_rmse = search3.cv_results_['mean_test_rmse'][best_fold_index_r2] 
    
for fold_index in range(cv):
    r2_fold = search3.cv_results_['split{}_test_r2'.format(fold_index)][best_fold_index_r2]
    rmse_fold = search3.cv_results_['split{}_test_rmse'.format(fold_index)][best_fold_index_r2]

    print(f"Fold {fold_index + 1} - R^2: {r2_fold}, RMSE: {rmse_fold}")

    test_indices_fold = test_indices[fold_index]

    print(f"Fold {fold_index + 1} - Test Indices: {test_indices_fold}", '\n')

print('\n', 'Average R2 score:', average_r2)
print('Average RMSE:', average_rmse)

best_gp3 = search3.best_estimator_

## GP for Latter retention (76~100 cycle)

In [ ]:
kernel4 = Matern()

param_dist4 = {'alpha': np.logspace(-10,0,11),
               'kernel__length_scale': np.logspace(-5,5,11),
               'kernel__nu': np.linspace(0.5,10,20)
              }


model_gp4 = GaussianProcessRegressor(kernel= kernel4,
                                 alpha = 'alpha',
                                 optimizer=None,
                                 random_state=0)

search4 = RandomizedSearchCV(model_gp4, param_distributions=param_dist4,
                             n_iter = n_iter, cv = cv, scoring={'r2': custom_r2_scorer,'rmse': custom_rmse_scorer},  
                             n_jobs=-1, verbose=1, refit='r2')

search4.fit(x_train, y_train4_std)

print(search4.best_params_, '\n')

best_fold_index_r2 = np.argmax(search4.cv_results_['mean_test_r2'])

kf = KFold(n_splits=cv)
test_indices = []

for _, test_index in kf.split(x_train):
    test_indices.append(test_index)

average_r2 = search4.cv_results_['mean_test_r2'][best_fold_index_r2]
average_rmse = search4.cv_results_['mean_test_rmse'][best_fold_index_r2] 
    
for fold_index in range(cv):
    r2_fold = search4.cv_results_['split{}_test_r2'.format(fold_index)][best_fold_index_r2]
    rmse_fold = search4.cv_results_['split{}_test_rmse'.format(fold_index)][best_fold_index_r2]

    print(f"Fold {fold_index + 1} - R^2: {r2_fold}, RMSE: {rmse_fold}")

    test_indices_fold = test_indices[fold_index]

    print(f"Fold {fold_index + 1} - Test Indices: {test_indices_fold}", '\n')

print('\n', 'Average R2 score:', average_r2)
print('Average RMSE:', average_rmse)

best_gp4 = search4.best_estimator_

# Find out the next experiment point with constraied EI

In [ ]:
warnings.filterwarnings('ignore')

best_mu1 = np.nanmin(y_train1_std)
best_mu1_inv = best_mu1 * std(y_train1, axis=0) + mean(y_train1, axis=0)
best_mu2 = np.nanmax(y_train2_std)
best_mu2_inv = best_mu2 * std(y_train2, axis=0) + mean(y_train2, axis=0)
best_mu3 = np.nanmax(y_train3_std)
best_mu3_inv = best_mu3 * std(y_train3, axis=0) + mean(y_train3, axis=0)
best_mu4 = np.nanmax(y_train4_std)
best_mu4_inv = best_mu4 * std(y_train4, axis=0) + mean(y_train4, axis=0)

def EI(a, best_gp1, best_gp2, best_gp3, best_gp4,
       best_mu1, best_mu2, best_mu3, best_mu4, e= 0): 
    
    X_new = search_space_final[int(a),:]
    X_new = X_new.reshape(1,-1)
    
    mu1, sigma1 = best_gp1.predict(X_new, return_std = True)
    z1 = np.zeros(mu1.shape)
    z11 = np.zeros(mu1.shape)
    z1[sigma1 > 0 ] = ((best_mu1 - mu1 - e)/sigma1)[sigma1 > 0]
    const1 = ((6 - mean(y_train1, axis=0)) / std(y_train1, axis=0))
    z11[sigma1 > 0 ] = ((mu1 - const1)/sigma1)[sigma1 > 0]
    EI1 = ((best_mu1 - mu1 - e) * norm.cdf(z1) + sigma1 * norm.pdf(z1))
    EI1_new = EI1 * (1 - norm.cdf(z11))
    
    
    mu2, sigma2 = best_gp2.predict(X_new, return_std = True)
    z2 = np.zeros(mu2.shape)
    z22 = np.zeros(mu2.shape)
    z2[sigma2 > 0 ] = ((mu2 - best_mu2 - e)/sigma2)[sigma2 > 0]
    const2 = ((80 - mean(y_train2, axis=0)) / std(y_train2, axis=0))           
    z22[sigma2 > 0 ] = ((mu2 - const2)/sigma2)[sigma2 > 0]
    EI2 = ((mu2 - best_mu2- e) * norm.cdf(z2) + sigma2 * norm.pdf(z2))
    EI2_new = EI2 * (norm.cdf(z22)) 
    
    
    mu3, sigma3 = best_gp3.predict(X_new, return_std = True)
    z3 = np.zeros(mu3.shape)
    z33 = np.zeros(mu3.shape)
    z3[sigma3 > 0 ] = ((mu3 - best_mu3 - e)/sigma3)[sigma3 > 0]
    const3 = ((148 - mean(y_train3, axis=0)) / std(y_train3, axis=0))           
    z33[sigma3 > 0 ] = ((mu3 - const3)/sigma3)[sigma3 > 0]
    EI3 = ((mu3 - best_mu3- e) * norm.cdf(z3) + sigma3 * norm.pdf(z3))
    EI3_new = EI3 * (norm.cdf(z33))
    
    
    mu4, sigma4 = best_gp4.predict(X_new, return_std = True)
    z4 = np.zeros(mu4.shape)
    z44 = np.zeros(mu4.shape)
    z4[sigma4 > 0 ] = ((mu4 - best_mu4 - e)/sigma4)[sigma4 > 0]
    const4 = ((90 - mean(y_train4, axis=0)) / std(y_train4, axis=0))           
    z44[sigma4 > 0 ] = ((mu4 - const4)/sigma4)[sigma4 > 0]
    EI4 = ((mu4 - best_mu4- e) * norm.cdf(z4) + sigma4 * norm.pdf(z4))
    EI4_new = EI4 * (norm.cdf(z44))

    
    return (EI1_new + 30*EI2_new + 20*EI3_new + 15*EI4_new)

EI_values = np.empty((search_space_final.shape[0],1))

for i in range(0, search_space_final.shape[0]):
    EI_value = EI(i, best_gp1, best_gp2, best_gp3, best_gp4,
                  best_mu1, best_mu2, best_mu3, best_mu4, e= 0)
    EI_values[i,0] = EI_value

result_new = np.nanmax(EI_values)
result_new_index = np.where(EI_values == result_new)[0]
print('---------------------------------------------------')
print('Best SET:', best_mu1_inv.tolist())
print('Best Retention:', best_mu2_inv.tolist())
print('Best Final discharge capacity (100th):', best_mu3_inv.tolist())
print('Best Latter retention (76~100 cycle):', best_mu4_inv.tolist(), '\n')

print("Objective function", result_new)
print("EI vlaue :", EI(result_new_index, best_gp1, best_gp2, best_gp3, best_gp4,
                  best_mu1, best_mu2, best_mu3, best_mu4, e= 0))
print('Next experiment point:', search_space_final[result_new_index, :])
pred1 = best_gp1.predict(search_space_final[result_new_index, :].reshape(1,-1))
pred1_inv = pred1 * std(y_train1, axis=0) + mean(y_train1, axis=0)
print('SET:', pred1_inv.tolist())
pred2 = best_gp2.predict(search_space_final[result_new_index, :].reshape(1,-1))
pred2_inv = pred2 * std(y_train2, axis=0) + mean(y_train2, axis=0)
print('Retention:', pred2_inv.tolist())
pred3 = best_gp3.predict(search_space_final[result_new_index, :].reshape(1,-1))
pred3_inv = pred3 * std(y_train3, axis=0) + mean(y_train3, axis=0)
print('Final discharge capacity (100th):', pred3_inv.tolist())
pred4 = best_gp4.predict(search_space_final[result_new_index, :].reshape(1,-1))
pred4_inv = pred4 * std(y_train4, axis=0) + mean(y_train4, axis=0)
print('Latter retention (76~100 cycle):', pred4_inv.tolist())
print('---------------------------------------------------\n')

print(EI(result_new_index, best_gp1, best_gp2, best_gp3, best_gp4,
         best_mu1, best_mu2, best_mu3, best_mu4, e= 0))

search_space_final = np.delete(search_space_final, result_new_index, axis = 0)
search_space_final.shape

# Gernerate additional experiment points for parallel experiment

In [ ]:
EI_values = np.empty((search_space_final.shape[0],1))

for i in range(0, search_space_final.shape[0]):
    EI_value = EI(i, best_gp1, best_gp2, best_gp3, best_gp4,
                  best_mu1, best_mu2, best_mu3, best_mu4, e= 0)
    EI_values[i,0] = EI_value


#-----------------------------------------------------------------------------------------------------------------

explore_level = 0.05
index_relevant = np.argpartition(EI_values, -int(explore_level * EI_values.shape[0]), axis=0)[::-1]
partitioned_area = search_space_final[index_relevant,:]
Top_1_EI = partitioned_area[0:int(explore_level * EI_values.shape[0])+1,:]

#-----------------------------------------------------------------------------------------------------------------
Std_values = np.empty((int(explore_level * EI_values.shape[0]),1))

for i in range(0, int(explore_level * EI_values.shape[0])):
    mu_1, std_1 = best_gp1.predict(Top_1_EI[i,:], return_std = True)
    mu_2, std_2 = best_gp2.predict(Top_1_EI[i,:], return_std = True)
    mu_3, std_3 = best_gp3.predict(Top_1_EI[i,:], return_std = True)
    mu_4, std_4 = best_gp4.predict(Top_1_EI[i,:], return_std = True)
    Total_std = (std_1**2 + std_2**2 + std_3**2 + std_4**2)**0.5
    Std_values[i,0] = Total_std

Next_point_index = Std_values.argsort(axis=0)[::-1]

add_num = 2 
Next_point_index = Next_point_index[:add_num,:]
Next_point = Top_1_EI[Next_point_index,:]
Next_point = np.squeeze(Next_point)

for i in range (0,add_num):
    print('Additional experiment points ', i+1, ' \n\n', Next_point[i,:], '\n')
    next1 = best_gp1.predict(Next_point[i,:].reshape(1,-1))
    next1_inv = next1 * std(y_train1, axis=0) + mean(y_train1, axis=0)
    print('SET:', next1_inv.tolist())
    next2 = best_gp2.predict(Next_point[i,:].reshape(1,-1))
    next2_inv = next2 * std(y_train2, axis=0) + mean(y_train2, axis=0)
    print('Retention:', next2_inv.tolist())
    next3 = best_gp3.predict(Next_point[i,:].reshape(1,-1))
    next3_inv = next3 * std(y_train3, axis=0) + mean(y_train3, axis=0)
    print('Final discharge capacity (100th):', next3_inv.tolist())
    next4 = best_gp4.predict(Next_point[i,:].reshape(1,-1))
    next4_inv = next4 * std(y_train4, axis=0) + mean(y_train4, axis=0)
    print('Latter retention (76~100 cycle):', next4_inv.tolist(),'\n\n')
  
    matching_rows = np.where((search_space_final == Next_point[i,:]).all(axis=1))
    mu_1, std_1 = best_gp1.predict(Next_point[i,:].reshape(1,-1), return_std = True)
    mu_2, std_2 = best_gp2.predict(Next_point[i,:].reshape(1,-1), return_std = True)
    mu_3, std_3 = best_gp3.predict(Next_point[i,:].reshape(1,-1), return_std = True)
    mu_4, std_4 = best_gp4.predict(Next_point[i,:].reshape(1,-1), return_std = True)
    Total_std = (std_1**2 + std_2**2 + std_3**2 + std_4**2)**0.5
    search_space_final = np.delete(search_space_final, matching_rows, axis = 0)
    print('-----------------------------------------------\n\n')
    
    
search_space_final.shape